<a href="https://colab.research.google.com/github/Vaibhav-Singh27/AI/blob/main/VAIBHAV_AI_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-flash-latest')

initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

chat = gemini_model.start_chat(
    history=[
        {'role': 'user', 'parts': [initial_system_instruction]},
        {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
    ]
)

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import os
import traceback
# Removed: from PIL import Image
# Removed: import io

try:
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Changed model to gemini-2.5-flash for multiturn chat support
    gemini_model = genai.GenerativeModel('gemini-2.5-flash')

    initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

    st.title("PIKA PIKA AI Chat") # Updated title as image generation is removed

    if "chat_session" not in st.session_state:
        st.session_state.chat_session = gemini_model.start_chat(
            history=[
                {'role': 'user', 'parts': [initial_system_instruction]},
                {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
            ]
        )

    for message in st.session_state.chat_session.history[2:]:
        if message.role == 'user':
            with st.chat_message("user"):
                st.markdown(message.parts[0])
        elif message.role == 'model':
            with st.chat_message("assistant"):
                # Removed image display logic, always display text
                st.markdown(message.parts[0])

    text_input = st.chat_input("Ask PIKA PIKA AI...") # Updated prompt as image generation is removed

    if text_input:
        final_message = text_input
    else:
        final_message = ""

    if final_message:
        with st.chat_message("user"):
            st.markdown(final_message)

        try:
            # Removed image generation request logic, only normal text conversation
            response = st.session_state.chat_session.send_message(final_message)
            with st.chat_message("assistant"):
                st.markdown(response.text)

        except Exception as e:
            st.error(f"An error occurred with Gemini model: {e}. Please check your API key and model configuration.")
            st.error(traceback.format_exc())

except Exception as e:
    st.error(f"An unhandled error occurred during Streamlit app execution: {e}")
    st.error(traceback.format_exc())

In [ ]:
!pip install -qqq streamlit

In [ ]:
# Install necessary libraries for audio transcription
!pip install -qqq SpeechRecognition pydub

In [ ]:
# Install st_audiorec for direct audio recording
!pip install -qqq st_audiorec

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import os
import traceback
import speech_recognition as sr
import io
import tempfile
import pydub
from PIL import Image # Added for image handling

try:
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Changed model to gemini-2.5-flash-image for multimodal capabilities
    gemini_model = genai.GenerativeModel('gemini-2.5-flash-image')

    initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries, including analyzing images if provided."

    st.title("PIKA PIKA AI Chat & Image Recognition") # Updated title again

    if "chat_session" not in st.session_state:
        st.session_state.chat_session = gemini_model.start_chat(
            history=[
                {'role': 'user', 'parts': [initial_system_instruction]},
                {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
            ]
        )

    for message in st.session_state.chat_session.history[2:]:
        if message.role == 'user':
            with st.chat_message("user"):
                for part in message.parts:
                    if isinstance(part, str):
                        st.markdown(part)
                    elif hasattr(part, 'mime_type') and 'image' in part.mime_type:
                        st.image(part.data, caption='Uploaded Image', use_column_width=True)
        elif message.role == 'model':
            with st.chat_message("assistant"):
                st.markdown(message.parts[0])

    transcribed_text_for_this_turn = ""
    uploaded_image_for_this_turn = None

    # --- Direct Audio Recording Section (not implemented yet, but keeping for future if alternative found) ---
    # st.subheader("Record your message:")
    # # Using a placeholder as st_audiorec failed
    # st.write("Direct audio recording is not available at the moment. Please use file upload or type your message.")

    # --- Audio File Upload Section ---
    st.subheader("Upload an audio file (WAV, MP3, FLAC) for transcription:")
    audio_file = st.file_uploader("", type=["wav", "mp3", "flac"], key="audio_uploader")

    if audio_file is not None:
        st.info("Transcribing audio, please wait...")
        try:
            suffix = os.path.splitext(audio_file.name)[1]
            with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp_file:
                tmp_file.write(audio_file.getvalue())
                temp_audio_path = tmp_file.name

            if suffix.lower() not in ['.wav', '.ogg'] and pydub:
                audio = pydub.AudioSegment.from_file(temp_audio_path)
                wav_path = tempfile.mktemp(suffix=".wav")
                audio.export(wav_path, format="wav")
                audio_source_path = wav_path
            else:
                audio_source_path = temp_audio_path

            recognizer = sr.Recognizer()
            with sr.AudioFile(audio_source_path) as source:
                audio_data = recognizer.record(source)
                transcribed_text_for_this_turn = recognizer.recognize_google(audio_data)

            if transcribed_text_for_this_turn:
                st.success("Transcription complete!")
                st.write(f"Transcribed Text: {transcribed_text_for_this_turn}")
            else:
                st.warning("Could not transcribe audio. Please try again.")

        except sr.UnknownValueError:
            st.error("Speech Recognition could not understand audio.")
        except sr.RequestError as e:
            st.error(f"Could not request results from Google Speech Recognition service; {e}")
        except Exception as e:
            st.error(f"An unexpected error occurred during audio transcription: {e}")
        finally:
            if 'temp_audio_path' in locals() and os.path.exists(temp_audio_path):
                os.remove(temp_audio_path)
            if 'wav_path' in locals() and os.path.exists(wav_path):
                os.remove(wav_path)
    # --- End Audio Input Section ---

    # --- Image Upload Section ---
    st.subheader("Upload an image for analysis:")
    image_file = st.file_uploader("", type=["png", "jpg", "jpeg"], key="image_uploader")

    if image_file is not None:
        uploaded_image_for_this_turn = Image.open(image_file)
        st.image(uploaded_image_for_this_turn, caption='Uploaded Image', use_column_width=True)
    # --- End Image Upload Section ---

    st.subheader("Type your message:")
    text_input = st.chat_input("Ask PIKA PIKA AI...")

    final_message_parts = []

    if transcribed_text_for_this_turn:
        final_message_parts.append(transcribed_text_for_this_turn)
    if text_input:
        final_message_parts.append(text_input)
    if uploaded_image_for_this_turn:
        final_message_parts.append(uploaded_image_for_this_turn)

    if final_message_parts:
        with st.chat_message("user"):
            for part in final_message_parts:
                if isinstance(part, str):
                    st.markdown(part)
                elif isinstance(part, Image.Image):
                    st.image(part, caption='User provided image', use_column_width=True)

        try:
            response = st.session_state.chat_session.send_message(final_message_parts)
            with st.chat_message("assistant"):
                st.markdown(response.text)

        except Exception as e:
            st.error(f"An error occurred with Gemini model: {e}. Please check your API key and model configuration.")
            st.error(traceback.format_exc())

except Exception as e:
    st.error(f"An unhandled error occurred during Streamlit app execution: {e}")
    st.error(traceback.format_exc())

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import os
import traceback
import speech_recognition as sr # Added for speech recognition
import io # Added for handling audio data
import tempfile # Added for temporary file creation
import pydub # Added for audio format handling

try:
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    gemini_model = genai.GenerativeModel('gemini-2.5-flash')

    initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

    st.title("PIKA PIKA AI Chat")

    if "chat_session" not in st.session_state:
        st.session_state.chat_session = gemini_model.start_chat(
            history=[
                {'role': 'user', 'parts': [initial_system_instruction]},
                {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
            ]
        )

    for message in st.session_state.chat_session.history[2:]:
        if message.role == 'user':
            with st.chat_message("user"):
                st.markdown(message.parts[0])
        elif message.role == 'model':
            with st.chat_message("assistant"):
                st.markdown(message.parts[0])

    # --- New Audio Input Section ---
    transcribed_text_for_this_turn = "" # Will hold text from uploaded audio for the current run

    audio_file = st.file_uploader("Upload an audio file (WAV, MP3, FLAC) for transcription", type=["wav", "mp3", "flac"])

    if audio_file is not None:
        st.info("Transcribing audio, please wait...")
        try:
            # Create a temporary file to store the audio
            suffix = os.path.splitext(audio_file.name)[1]
            with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp_file:
                tmp_file.write(audio_file.getvalue())
                temp_audio_path = tmp_file.name

            # Convert to WAV for SpeechRecognition if not already WAV
            if suffix.lower() not in ['.wav', '.ogg'] and pydub: # pydub can handle more formats
                audio = pydub.AudioSegment.from_file(temp_audio_path)
                wav_path = tempfile.mktemp(suffix=".wav")
                audio.export(wav_path, format="wav")
                audio_source_path = wav_path
            else:
                audio_source_path = temp_audio_path

            recognizer = sr.Recognizer()
            with sr.AudioFile(audio_source_path) as source:
                audio_data = recognizer.record(source)
                transcribed_text_for_this_turn = recognizer.recognize_google(audio_data)

            if transcribed_text_for_this_turn:
                st.success("Transcription complete!")
                st.write(f"Transcribed Text: {transcribed_text_for_this_turn}") # Display transcribed text immediately
            else:
                st.warning("Could not transcribe audio. Please try again.")

        except sr.UnknownValueError:
            st.error("Speech Recognition could not understand audio.")
        except sr.RequestError as e:
            st.error(f"Could not request results from Google Speech Recognition service; {e}")
        except Exception as e:
            st.error(f"An unexpected error occurred during audio transcription: {e}")
        finally:
            # Clean up temporary files
            if 'temp_audio_path' in locals() and os.path.exists(temp_audio_path):
                os.remove(temp_audio_path)
            if 'wav_path' in locals() and os.path.exists(wav_path):
                os.remove(wav_path)
    # --- End Audio Input Section ---

    text_input = st.chat_input("Ask PIKA PIKA AI...")

    final_message = ""
    if text_input:
        final_message = text_input
    elif transcribed_text_for_this_turn: # Prioritize transcribed audio if just uploaded
        final_message = transcribed_text_for_this_turn

    if final_message:
        with st.chat_message("user"):
            st.markdown(final_message)

        try:
            response = st.session_state.chat_session.send_message(final_message)
            with st.chat_message("assistant"):
                st.markdown(response.text)

        except Exception as e:
            st.error(f"An error occurred with Gemini model: {e}. Please check your API key and model configuration.")
            st.error(traceback.format_exc())

except Exception as e:
    st.error(f"An unhandled error occurred during Streamlit app execution: {e}")
    st.error(traceback.format_exc())

In [ ]:
import os
import time
import re
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError

# GOOGLE_API_KEY is already set in the environment and configured for genai
# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY # This line is not necessary here

# Kill any existing streamlit processes
!pkill -f streamlit || true

# Kill any existing ngrok processes to ensure a clean start
print('Stopping any existing ngrok processes...')
ngrok.kill()
print('Existing ngrok processes stopped.')

# Start streamlit in the background
print('Starting Streamlit app...')
!setsid streamlit run app.py --server.port 8501 > streamlit_app.log 2>&1 &
time.sleep(5) # Give Streamlit some time to start

# NGROK_AUTH_TOKEN is already set and authenticated in a previous cell
# No need to check NGROK_AUTH_TOKEN or set auth token again here.

try:
    tunnel = None
    for i in range(3):
        try:
            print(f'Attempt {i+1} to establish ngrok tunnel...')
            tunnel = ngrok.connect(8501)
            break
        except PyngrokNgrokError as e:
            print(f'Ngrok tunnel connection failed: {e}. Retrying in 5 seconds...')
            time.sleep(5)

    if tunnel:
        ngrok_url = tunnel.public_url
        print(f"Streamlit app available at: {ngrok_url}")
    else:
        print("Failed to establish ngrok tunnel after multiple attempts.")

except Exception as e:
    print(f"An error occurred while setting up ngrok tunnel: {e}")
    print("Please check if ngrok is installed and your NGROK_AUTH_TOKEN is correct.")

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"Click [here]({ngrok_url}) to access your Streamlit app."))

In [ ]:
!pip install -qqq pyngrok

In [ ]:
from google.colab import userdata
import os

raw_ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')

# Extract the actual token string
# This handles cases like 'export NGROK_AUTHTOKEN=3DYOUR_TOKEN' or 'YOUR_TOKEN'
if raw_ngrok_auth_token and '=' in raw_ngrok_auth_token:
    clean_ngrok_auth_token = raw_ngrok_auth_token.split('=')[-1]
else:
    clean_ngrok_auth_token = raw_ngrok_auth_token

os.environ["NGROK_AUTH_TOKEN"] = clean_ngrok_auth_token

!ngrok authtoken {clean_ngrok_auth_token}
print('ngrok authenticated successfully.')